In [2]:
# ============================================================
# Figure e/f/g QC Plot
#
# Author: Wang Ning
#
# 功能：
# Figure e : Replicate correlation
# Figure f : RNA UMI violin
# Figure g : Barcode rank plot

#for fig e
# NormalizeData后的 data slot
# ↓
# 按 Batch 提取两个真实重复
# ↓
# 计算每个基因在两个重复中的平均表达
# ↓
# 过滤两个重复都表达的基因
# ↓
# Pearson相关性
# ↓
# 绘制论文风格相关性图
# ============================================================

library(Seurat)
library(ggplot2)
library(Matrix)
library(patchwork)

# ============================================================
# 参数设置
# ============================================================

prefix <- "tor"

rds_file <- "/data/work/RNA_AraSeed/rds/tor/tor_recluster_annotated.rds"

output_dir <- "/data/work/RNA_AraSeed/QC/tor/new_QC_sumary"

dir.create(
  output_dir,
  recursive = TRUE,
  showWarnings = FALSE
)

# ============================================================
# 读取对象
# ============================================================

cat("Loading Seurat object...\n")

obj <- readRDS(rds_file)

# ============================================================
# 提取Batch信息
# ============================================================

batch_levels <- unique(as.character(obj$Batch))

print(table(obj$Batch))

if(length(batch_levels) != 2){
  stop("Figure e/f/g需要恰好两个Batch")
}

batch1 <- batch_levels[1]
batch2 <- batch_levels[2]

cat("Rep1:", batch1, "\n")
cat("Rep2:", batch2, "\n")

# ============================================================
# 获取counts矩阵
# genes × cells
# ============================================================

counts <- GetAssayData(
  obj,
  slot = "counts"
)

# ============================================================
# 获取两个重复对应细胞
# ============================================================

cells_rep1 <- colnames(obj)[obj$Batch == batch1]

cells_rep2 <- colnames(obj)[obj$Batch == batch2]

cat("Rep1 cells:", length(cells_rep1), "\n")
cat("Rep2 cells:", length(cells_rep2), "\n")

# ============================================================
# Figure e
# Gene expression correlation
## use raw counts 
# ============================================================

cat("Calculating correlation...\n")

gene_rep1 <- Matrix::rowSums(
  counts[, cells_rep1]
)

gene_rep2 <- Matrix::rowSums(
  counts[, cells_rep2]
)

df_cor <- data.frame(
  rep1 = log10(gene_rep1 + 1),
  rep2 = log10(gene_rep2 + 1)
)

cor_res <- cor.test(
  df_cor$rep1,
  df_cor$rep2,
  method = "pearson"
)

R_value <- round(
  as.numeric(cor_res$estimate),
  2
)

P_value <- format(
  cor_res$p.value,
  scientific = TRUE,
  digits = 2
)

p_cor <- ggplot(
  df_cor,
  aes(rep1, rep2)
) +
  geom_point(
    size = 0.3,
    alpha = 0.4
  ) +
  geom_abline(
    slope = 1,
    intercept = 0,
    linetype = "dashed"
  ) +
  annotate(
    "text",
    x = min(df_cor$rep1),
    y = max(df_cor$rep2),
    hjust = 0,
    vjust = 1,
    size = 5,
    label = paste0(
      "R = ",
      R_value,
      "\n",
      "p = ",
      P_value
    )
  ) +
  theme_bw(base_size = 14) +
  labs(
    title = prefix,
    x = paste0(batch1, " log10(UMI+1)"),
    y = paste0(batch2, " log10(UMI+1)")
  )

ggsave(
  file.path(
    output_dir,
    paste0(prefix, "_FigureE_Correlation.pdf")
  ),
  p_cor,
  width = 5,
  height = 5
)

##
# ============================================================
# Figure e
# Correlation between biological replicates
#
# Each point represents one gene.
#
# X-axis:
# Total UMI counts of a gene in Replicate 1
#
# Y-axis:
# Total UMI counts of a gene in Replicate 2
#
# Only genes expressed in both replicates are retained.
# ============================================================

cat("Calculating correlation...\n")

gene_rep1 <- Matrix::rowSums(
    counts[, cells_rep1]
)

gene_rep2 <- Matrix::rowSums(
    counts[, cells_rep2]
)

# ============================================================
# Remove genes with zero expression
# in either replicate
# ============================================================

keep <- gene_rep1 > 5 &  #数值代表筛选严格度
        gene_rep2 > 5

cat("Total genes:",
    length(gene_rep1),
    "\n")

cat("Retained genes:",
    sum(keep),
    "\n")

cat("Filtered genes:",
    sum(!keep),
    "\n")

# ============================================================
# Log-transform
# ============================================================

df_cor <- data.frame(
    rep1 = log10(
        gene_rep1[keep] + 1
    ),
    rep2 = log10(
        gene_rep2[keep] + 1
    )
)

# ============================================================
# Pearson correlation
# ============================================================

cor_res <- cor.test(
    df_cor$rep1,
    df_cor$rep2,
    method = "pearson"
)

R_value <- round(
    cor_res$estimate,
    2
)

P_value <- format(
    cor_res$p.value,
    scientific = TRUE,
    digits = 2
)

p_cor <- ggplot(
    df_cor,
    aes(rep1, rep2)
) +
    geom_point(
        size = 0.3,
        alpha = 0.4
    ) +
    geom_abline(
        slope = 1,
        intercept = 0,
        linetype = "dashed"
    ) +
    annotate(
        "text",
        x = 0.2,
        y = 5.8,
        hjust = 0,
        vjust = 1,
        size = 5,
        label = paste0(
            "R = ",
            R_value,
            "\n",
            "p = ",
            P_value
        )
    ) +

    # ========================================================
    # 关键修改：使用coord_cartesian代替limits
    # ========================================================
    coord_cartesian(
        xlim = c(0, 6),
        ylim = c(0, 6)
    ) +

    scale_x_continuous(
        breaks = 0:6
    ) +

    scale_y_continuous(
        breaks = 0:6
    ) +

    theme_bw(base_size = 14) +

    labs(
        title = prefix,
        x = paste0(
            batch1,
            " log10(UMI+1)"
        ),
        y = paste0(
            batch2,
            " log10(UMI+1)"
        )
    )

ggsave(
    file.path(
        output_dir,
        paste0(
            prefix,
            "_FigureE_Correlation_filtered_strict.pdf"
        )
    ),
    p_cor,
    width = 5,
    height = 5
)


# ============================================================
# Figure f
# RNA UMI violin
# ============================================================

cat("Drawing violin plot...\n")

df_violin <- data.frame(
  UMI = c(
    obj$nCount_RNA[obj$Batch == batch1],
    obj$nCount_RNA[obj$Batch == batch2]
  ),
  Batch = c(
    rep(
      batch1,
      sum(obj$Batch == batch1)
    ),
    rep(
      batch2,
      sum(obj$Batch == batch2)
    )
  )
)

p_violin <- ggplot(
  df_violin,
  aes(
    Batch,
    UMI
  )
) +
  geom_violin(
    fill = "#00A65A",
    color = "black"
  ) +
  scale_y_log10() +
  theme_bw(base_size = 14) +
  labs(
    y = "RNA UMI number",
    x = NULL
  )

ggsave(
  file.path(
    output_dir,
    paste0(prefix, "_FigureF_Violin.pdf")
  ),
  p_violin,
  width = 3,
  height = 5
)

# ============================================================
# Figure g
# Barcode Rank Plot
# ============================================================

cat("Drawing barcode rank plot...\n")

plot_knee <- function(umi_vector, sample_name){

  counts_sorted <- sort(
    umi_vector,
    decreasing = TRUE
  )

  rank_df <- data.frame(
    Rank = seq_along(counts_sorted),
    Count = counts_sorted
  )

  ggplot(
    rank_df,
    aes(Rank, Count)
  ) +
    geom_line(
      color = "#00A65A",
      linewidth = 1
    ) +
    scale_x_log10() +
    scale_y_log10() +
    theme_bw(base_size = 12) +
    labs(
      title = sample_name,
      x = "Barcode Rank",
      y = "Counts"
    )
}

p1 <- plot_knee(
  obj$nCount_RNA[obj$Batch == batch1],
  batch1
)

p2 <- plot_knee(
  obj$nCount_RNA[obj$Batch == batch2],
  batch2
)

p_knee <- p1 + p2

ggsave(
  file.path(
    output_dir,
    paste0(prefix, "_FigureG_KneePlot.pdf")
  ),
  p_knee,
  width = 10,
  height = 4
)

# ============================================================
# 输出统计信息
# ============================================================

summary_df <- data.frame(
  Sample = prefix,
  Batch1 = batch1,
  Batch2 = batch2,
  Cells_Rep1 = length(cells_rep1),
  Cells_Rep2 = length(cells_rep2),
  Correlation_R = R_value
)

write.csv(
  summary_df,
  file.path(
    output_dir,
    paste0(prefix, "_QC_summary.csv")
  ),
  row.names = FALSE
)

cat("\n")
cat("=====================================\n")
cat("Finished\n")
cat("=====================================\n")

Loading Seurat object...

torC250410005 torC250410006 
        17742         18378 
Rep1: torC250410005 
Rep2: torC250410006 
Rep1 cells: 17742 
Rep2 cells: 18378 
Calculating correlation...
Calculating correlation...
Total genes: 32347 
Retained genes: 26784 
Filtered genes: 5563 
Drawing violin plot...
Drawing barcode rank plot...

Finished
